# **TaniMol: 05 - Activity Analysis**

This notebook maps the relationship between structural similarity and biological activity. Using the Butina clusters from notebook 04 and the pIC50 values from the preprocessed dataset, we evaluate whether structurally similar molecules exhibit similar potency against PARP1.

The analysis consists of four components:
1. **Within-cluster activity distributions** — do molecules in the same structural cluster have consistent pIC50 values?
2. **Activity cliff detection** — pairs of structurally similar molecules (Tanimoto > 0.8) with dramatically different activity (|ΔpIC50| > 2, i.e. >100-fold potency difference)
3. **SALI (Structure-Activity Landscape Index)** — a continuous metric that ranks how "dramatic" each activity cliff is, without arbitrary thresholds
4. **Similarity-activity correlation (Spearman)** — a global statistical test of the SAR hypothesis: do similar molecules tend to have similar activity?

> **Note:** This analysis uses exclusively **Morgan (ECFP4)** fingerprints, the primary representation in this project. MACCS and RDKit fingerprints are available for methodological comparison but are not evaluated here.

**Input:** Clustering results (`.pkl`) and preprocessed activity data (`.csv`)  
**Output:** Cluster activity statistics, activity cliff pairs, SALI scores, Spearman correlation

In [1]:
from src.config import PROCESSED_DIR
from src.activity_analysis import (
    within_cluster_activity_distributions,
    activity_cliffs,
    sali,
    similarity_activity_correlation
)
import pandas as pd
import numpy as np
import pickle

### **1. Load Data**

Load the clustering results (similarity matrices, cluster dictionaries, singleton lists) and the cleaned bioactivity dataset. Extract pIC50 values as a NumPy array and precompute the pairwise |ΔpIC50| matrix — this single NxN matrix will be reused across all four analyses.

In [2]:
with open(f"{PROCESSED_DIR}/clustering_results.pkl", "rb") as f:
    results = pickle.load(f)
    
morgan_sim = results["morgan"]["similarity_matrix"]
morgan_clusters = results["morgan"]["clusters"]
morgan_singletons = results["morgan"]["singletons"]

df = pd.read_csv(f"{PROCESSED_DIR}/cleaned_activities.csv")
pic50 = df["pchembl_value"].values
delta_pic50 = np.abs(pic50[:, None] - pic50[None, :])

print(f"Loaded {len(df)} molecules.")
print(f"Similarity matrix: {morgan_sim.shape}")
print(f"ΔpIC50 matrix: {delta_pic50.shape}")
print(f"Clusters: {len(morgan_clusters)}")
print(f"Singletons: {len(morgan_singletons)}")

Loaded 3324 molecules.
Similarity matrix: (3324, 3324)
ΔpIC50 matrix: (3324, 3324)
Clusters: 339
Singletons: 370


### **2. Within-Cluster Activity Distributions**

For each Butina cluster, extract the pIC50 values of all member molecules and compute summary statistics (mean, median, standard deviation, min, max). Low standard deviation within a cluster confirms the SAR hypothesis: structurally similar molecules have similar biological activity. High variance suggests the structure-activity relationship breaks down within that scaffold, potentially indicating activity cliffs.

Results are displayed as a table sorted by cluster size (largest first).

In [3]:
cluster_stats = within_cluster_activity_distributions(morgan_clusters, pic50)

stats_df = pd.DataFrame.from_dict(cluster_stats, orient="index")
stats_df.index.name = "centroid_idx"
stats_df = stats_df.sort_values("n", ascending=False)
stats_df

,n,mean,median,std,min,max
centroid_idx,,,,,,
401,173,7.972890,8.150,1.046534,5.30,10.00
1374,108,7.699907,7.950,0.687097,5.76,8.71
2355,95,8.228000,8.220,0.474973,6.40,9.47
2568,73,8.118630,8.090,0.470460,7.22,9.29
704,51,8.310588,8.430,0.503468,6.70,9.15
...,...,...,...,...,...,...
427,2,6.200000,6.200,0.100000,6.10,6.30
278,2,5.550000,5.550,0.550000,5.00,6.10
225,2,5.885000,5.885,0.675000,5.21,6.56


### **3. Activity Cliffs**

Activity cliffs are pairs of molecules that are structurally very similar (Tanimoto > 0.8) but have dramatically different biological activity (|ΔpIC50| > 2.0, meaning a >100-fold difference in potency). These pairs are scientifically the most interesting — they point to specific structural modifications that have an outsized impact on target binding.

Detection uses vectorized NumPy operations on the precomputed similarity and ΔpIC50 matrices, avoiding any Python loops.

In [4]:
cliffs = activity_cliffs(morgan_sim, delta_pic50)
cliffs_df = pd.DataFrame(cliffs)
print(f"Found {len(cliffs)} activity cliffs")
cliffs_df.head(20)

Found 37 activity cliffs


,mol_i,mol_j,similarity,delta_pic50
0,877,884,0.805556,4.05
1,1786,1788,0.838710,3.45
2,2496,2655,0.830508,3.27
3,2602,2605,0.811881,2.54
4,1754,1757,0.819444,2.52
5,1779,1789,0.828125,2.45
6,233,240,0.833333,2.44
7,2599,2605,0.820000,2.43
8,1779,1781,0.868852,2.41
9,2509,2520,0.840000,2.41


### **4. SALI (Structure-Activity Landscape Index)**

While activity cliffs use binary thresholds (similar/not similar, cliff/not cliff), SALI provides a **continuous score** for every pair of molecules:

$$SALI(i,j) = \frac{|\Delta pIC50|}{1 - Tanimoto}$$

SALI amplifies cases where *very* similar molecules have different activities. Two pairs may both pass the activity cliff threshold, but SALI distinguishes which one is more dramatic. For example, a pair with Tanimoto = 0.95 and |ΔpIC50| = 2.1 scores SALI = 42.0 — three times higher than a pair with Tanimoto = 0.85 and the same |ΔpIC50| (SALI = 14.0).

Note: when Tanimoto = 1.0 (identical fingerprints), division by zero is handled by setting SALI to 0.

In [5]:
sali_matrix, sali_values = sali(morgan_sim, delta_pic50)

print(f"Max SALI: {sali_values.max():.1f}")
print(f"Mean SALI: {sali_values.mean():.2f}")
print(f"Median SALI: {np.median(sali_values):.2f}")
print(f"Pairs with SALI > 50: {(sali_values > 50).sum()}")

Max SALI: 72.8
Mean SALI: 1.56
Median SALI: 1.33
Pairs with SALI > 50: 4


### **Top SALI Pairs — Biggest Activity Cliffs**

A high SALI score means a tiny structural change causes a disproportionately large shift in potency. Here are extracted the most extreme cases (SALI > 50) with their SMILES to identify which specific structural modifications drive the biggest activity changes against the target.


In [6]:
i_idx, j_idx = np.triu_indices_from(sali_matrix, k=1)
top_mask = sali_values > 50
top_pairs = pd.DataFrame({
    "mol_i": i_idx[top_mask],
    "mol_j": j_idx[top_mask],
    "similarity": morgan_sim[i_idx[top_mask], j_idx[top_mask]],
    "delta_pic50": delta_pic50[i_idx[top_mask], j_idx[top_mask]],
    "sali": sali_values[top_mask],
}).sort_values("sali", ascending=False)

top_pairs["smiles_i"] = top_pairs["mol_i"].map(lambda x: df.iloc[x]["canonical_smiles"])
top_pairs["smiles_j"] = top_pairs["mol_j"].map(lambda x: df.iloc[x]["canonical_smiles"])
top_pairs

,mol_i,mol_j,similarity,delta_pic50,sali,smiles_i,smiles_j
1,219,220,0.980769,1.40,72.799948,O=C1N=C(CCCN2CC=C(c3ccccc3)CC2)NC2CCCCC12,O=C1N=C(CCCN2CC=C(c3ccccc3)CC2)NC2CCCCCC12
3,2529,2530,0.986486,0.87,64.380038,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1CCn2c(...,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1CCn2c(...
0,19,21,0.981818,1.00,55.000052,O=C(CCn1c2c(c(=O)[nH]c1=O)CCCC2)NCC(=O)N1CCN(c...,O=C(CCn1c2c(c(=O)[nH]c1=O)CCCCC2)NCC(=O)N1CCN(...
2,392,393,0.982456,0.88,50.160021,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1CCN(C(...,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1CCN(C(...


### **5. Similarity-Activity Correlation (Spearman)**

The global SAR hypothesis states: *structurally similar molecules tend to have similar biological activity*. Spearman correlation tests this statistically by comparing two parallel vectors extracted from the upper triangle of the NxN matrices:
- Vector 1: all pairwise Tanimoto similarity values
- Vector 2: all corresponding |ΔpIC50| values

A **negative** Spearman ρ confirms SAR -> higher similarity correlates with smaller activity differences. A ρ near zero means structural similarity does not predict activity for this dataset.

In [7]:
rho, p_value = similarity_activity_correlation(morgan_sim, delta_pic50)

print(f"Spearman ρ = {rho:.4f}")
print(f"p-value = {p_value:.3f}")

if rho < -0.1 and p_value < 0.05:
    print("SAR confirmed: similar structures tend to have similar activity.")
elif abs(rho) < 0.1:
    print("No clear SAR: structural similarity does not predict activity.")
else:
    print("Unexpected positive correlation.")

Spearman ρ = -0.1149
p-value = 0.000
SAR confirmed: similar structures tend to have similar activity.
